# Arg KPI Government: presidentes y ministros de Economía

**Autor:** Renzo Gutierrez  
**Período:** 1 de enero de 2000 hasta la fecha de ejecución

## Objetivo

Construir un pipeline ETL que genere el contexto político y económico diario de Argentina:

1. Extraer el listado oficial de ministros del área económica.
2. Curar los períodos presidenciales publicados por Casa Rosada.
3. Normalizar cargos, nombres y fechas.
4. Crear tablas de períodos en Bronze.
5. Generar una dimensión diaria en Silver, con una fila por fecha.
6. Guardar los datasets como tablas Delta Lake sobre MinIO.

La tabla diaria está diseñada para unirse posteriormente con `Arg_KPI_USD` mediante la columna `date`. La tercera notebook podrá utilizar esa unión para analizar variaciones del dólar por presidente, ministro y cambio de gestión, y preparar un waterfall chart.


## 1. Diseño y fuentes

### Ministros de Economía

El Centro de Documentación e Información del Ministerio de Economía publica un CSV abierto con:

- denominación del cargo;
- nombre del funcionario;
- fecha de designación y cese;
- normas de alta y baja;
- referencias bibliográficas.

Fuente: https://cdi.mecon.gob.ar/ministros-de-economia

### Presidentes

Casa Rosada publica la nómina histórica y las páginas oficiales de cada presidencia, pero no ofrece un CSV homogéneo. Debido a que el período analizado contiene pocos mandatos, se utiliza una dimensión curada y auditable dentro de la notebook.

Fuente: https://www.casarosada.gob.ar/nuestro-pais/presidentes

### Capas de almacenamiento

| Capa | Tabla | Granularidad |
|---|---|---|
| Bronze | `presidential_terms` | Un registro por mandato |
| Bronze | `economy_minister_terms` | Un registro por gestión ministerial |
| Silver | `government_context_daily` | Un registro por día |

Los períodos se modelan como intervalos **cerrados por izquierda y abiertos por derecha**: `valid_from <= date < valid_to_exclusive`. Esto evita superposiciones en los días de cambio de autoridad.


## 2. Dependencias

Se utilizan `requests` para la extracción, `pandas` para la transformación, `pyarrow` para la interoperabilidad y `deltalake` para almacenamiento transaccional.


In [ ]:
%pip install -q requests pandas pyarrow deltalake matplotlib

In [ ]:
import io
import os
import unicodedata
import warnings
from datetime import date, timedelta
from getpass import getpass
from typing import Any

import matplotlib.pyplot as plt
import pandas as pd
import pyarrow as pa
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

from deltalake import DeltaTable, write_deltalake

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

## 3. Configuración

La notebook utiliza el mismo rango temporal que `Arg_KPI_USD`. Las credenciales de MinIO no se almacenan en el archivo: se leen desde variables de entorno o se solicitan al ejecutar.


In [ ]:
START_DATE = date(2000, 1, 1)
END_DATE = date.today()

MINISTERS_CSV_URL = (
    "https://cdi.mecon.gob.ar/vertical-timeline/js/ministros.csv"
)
REQUEST_TIMEOUT = (8, 30)  # segundos: conexión y lectura
VACUUM_RETENTION_HOURS = 168  # 7 días


def get_setting(name: str, prompt: str, secret: bool = False) -> str:
    """Obtiene una configuración del entorno o la solicita al usuario."""
    value = os.getenv(name)
    if value:
        return value
    return getpass(prompt) if secret else input(prompt).strip()


MINIO_ENDPOINT = get_setting("MINIO_ENDPOINT", "Endpoint de MinIO: ")
MINIO_ACCESS_KEY = get_setting("MINIO_ACCESS_KEY", "Access key de MinIO: ", secret=True)
MINIO_SECRET_KEY = get_setting("MINIO_SECRET_KEY", "Secret key de MinIO: ", secret=True)
MINIO_BUCKET = get_setting("MINIO_BUCKET", "Bucket de MinIO: ")

storage_options = {
    "AWS_ENDPOINT_URL": MINIO_ENDPOINT,
    "AWS_ACCESS_KEY_ID": MINIO_ACCESS_KEY,
    "AWS_SECRET_ACCESS_KEY": MINIO_SECRET_KEY,
    "AWS_ALLOW_HTTP": "true",
    "aws_conditional_put": "etag",
    "AWS_S3_ALLOW_UNSAFE_RENAME": "true",
}

# Bronze conserva los períodos normalizados cercanos a sus fuentes.
BRONZE_BASE_PATH = (
    f"s3://{MINIO_BUCKET}/lakehouse/bronze/arg_government_context"
)
PRESIDENTIAL_TERMS_PATH = f"{BRONZE_BASE_PATH}/presidential_terms"
ECONOMY_MINISTER_TERMS_PATH = f"{BRONZE_BASE_PATH}/economy_minister_terms"

# Silver contiene la dimensión diaria derivada y lista para análisis.
SILVER_BASE_PATH = (
    f"s3://{MINIO_BUCKET}/lakehouse/silver/arg_government_context"
)
GOVERNMENT_DAILY_PATH = f"{SILVER_BASE_PATH}/government_context_daily"

print(f"Rango del pipeline: {START_DATE} a {END_DATE}")
print(f"Bronze: {BRONZE_BASE_PATH}")
print(f"Silver: {SILVER_BASE_PATH}")

## 4. Cliente HTTP

La descarga del CSV incorpora reintentos para límites de tráfico y fallos transitorios del servidor. Como el sitio oficial puede quedar temporalmente inaccesible, la notebook incluye una copia de respaldo de los registros necesarios para el período analizado. La fuente en línea continúa siendo la opción primaria; el respaldo solo evita que el pipeline completo falle por indisponibilidad externa.


In [ ]:
def build_http_session() -> requests.Session:
    """Crea una sesión HTTP con reintentos y backoff exponencial."""
    retry_strategy = Retry(
        total=2,
        connect=1,
        backoff_factor=1,
        status_forcelist=(429, 500, 502, 503, 504),
        allowed_methods=frozenset(["GET"]),
    )

    adapter = HTTPAdapter(max_retries=retry_strategy)
    session = requests.Session()
    session.mount("https://", adapter)
    return session


HTTP_SESSION = build_http_session()


# Snapshot mínimo de respaldo obtenido del mismo CSV oficial.
# Se actualiza la información desde Internet siempre que la fuente responde.
MINISTERS_FALLBACK_AS_OF = "2026-09-27"
MINISTERS_FALLBACK_CSV = """Ministerio;Ministro;Fecha desde;Norma de alta;Fecha hasta;Norma de cese
Ministerio de Economía;CAPUTO, Luis Andrés;10.12.2023;;;
Ministro de Economía;MASSA, Tomás Sergio;03.08.2022;;07.12.2023;
Ministra de Economía;BATAKIS, Silvina Aida;04.07.2022;;03.08.2022;
Ministro de Economía;GUZMÁN, Martín Maximiliano;10.12.2019;;04.07.2022;
Ministro de Hacienda;LACUNZA, Jorge Roberto Hernán;20.08.2019;;09.12.2019;
Ministro de Hacienda;DUJOVNE, Nicolás;02.01.2017;;20.08.2019;
Ministro de Hacienda y Finanzas Públicas;PRAT GAY, Alfonso;10.12.2015;;31.12.2016;
Ministro de Economía y Finanzas Públicas;KICILLOF, Axel;20.11.2013;;10.12.2015;
Ministro de Economía y Finanzas Públicas;LORENZINO, Hernán Gaspar;10.12.2011;;20.11.2013;
Ministro de Economía y Finanzas Públicas;BOUDOU, Amado;10.07.2009;;10.12.2011;
Ministro de Economía y Finanzas Públicas;FERNANDEZ, Carlos Rafael;26.11.2008;;10.07.2009;
Ministro de Economía y Producción;FERNANDEZ, Carlos Rafael;25.04.2008;;26.11.2008;
Ministro de Economía y Producción;LOUSTEAU, Martín;10.12.2007;;25.04.2008;
Ministro de Economía y Producción;PEIRANO, Miguel Gustavo;17.07.2007;;10.12.2007;
Ministra de Economía y Producción;MICELI, Felisa Josefina;01.12.2005;;17.07.2007;
Ministro de Economía y Producción;LAVAGNA, Roberto;26.04.2002;;28.11.2005;
Ministro de Economía;REMES LENICOV, Jorge Luis;07.01.2002;;26.04.2002;
Secretario de Hacienda e Interino de Finanzas e Ingresos Públicos;FRIGERI, Rodolfo;23.12.2001;;07.01.2002;
Mtro. de Infraestructura y Vivienda, Interino de Economía, Desarrollo Social y Medio Ambiente, de Salud, de Trabajo Empleo y formación de recursos humanos y de Seguridad Social;CAPITANICH, Jorge Milton;21.12.2001;;22.12.2001;
Mtro. Economía y Mtro. de Infraestructura y Vivienda, Sin perjuicio de sus funciones como Secretario General de la Presidencia;GALLO, Nicolas Vicente;20.12.2001;;21.12.2001;
Mtro.Economía;CAVALLO, Domingo Felipe;20.03.2001;;20.12.2001;
Mtro.Economía;LOPEZ MURPHY, Ricardo Hipólito;05.03.2001;;20.03.2001;
Mtro.Economía;MACHINEA, José Luis;10.12.1999;;05.03.2001;
"""


def download_ministers_csv() -> bytes:
    """Descarga la fuente oficial o utiliza el snapshot si no está disponible."""
    try:
        response = HTTP_SESSION.get(
            MINISTERS_CSV_URL,
            timeout=REQUEST_TIMEOUT,
        )
        response.raise_for_status()
        print("Fuente de ministros: CSV oficial en línea")
        return response.content
    except requests.RequestException as error:
        warnings.warn(
            "El sitio oficial no respondió. "
            f"Se utilizará el snapshot oficial del {MINISTERS_FALLBACK_AS_OF}. "
            f"Detalle técnico: {error}",
            RuntimeWarning,
        )
        return MINISTERS_FALLBACK_CSV.encode("latin-1")

## 5. Extract: períodos presidenciales

Los períodos se registran con `valid_to_exclusive`, es decir, la fecha en la que comienza la siguiente presidencia. Para la presidencia vigente se utiliza el día posterior a `END_DATE`.

Se incluyen las autoridades provisionales de la crisis institucional de 2001 porque forman parte del contexto político del período analizado.


In [ ]:
def extract_presidential_terms(end_date: date = END_DATE) -> pd.DataFrame:
    """Construye la dimensión curada de presidencias desde fuentes oficiales."""
    records = [
        {
            "president": "Fernando de la Rúa",
            "valid_from": "1999-12-10",
            "valid_to_exclusive": "2001-12-21",
            "political_space": "Alianza / UCR",
            "mandate_type": "Constitucional",
            "presidential_term": 1,
            "source_url": "https://www.casarosada.gob.ar/nuestro-pais/presidentes/47090-fernando-de-la-rua-1999-2001",
        },
        {
            "president": "Federico Ramón Puerta",
            "valid_from": "2001-12-21",
            "valid_to_exclusive": "2001-12-23",
            "political_space": "Partido Justicialista",
            "mandate_type": "Provisional",
            "presidential_term": 1,
            "source_url": "https://www.casarosada.gob.ar/nuestro-pais/presidentes/47089-federico-ramon-puerta-2001",
        },
        {
            "president": "Adolfo Rodríguez Saá",
            "valid_from": "2001-12-23",
            "valid_to_exclusive": "2001-12-31",
            "political_space": "Partido Justicialista",
            "mandate_type": "Designado por Asamblea Legislativa",
            "presidential_term": 1,
            "source_url": "https://www.casarosada.gob.ar/nuestro-pais/presidentes/47088-adolfo-rodriguez-saa-2001",
        },
        {
            "president": "Eduardo Camaño",
            "valid_from": "2001-12-31",
            "valid_to_exclusive": "2002-01-02",
            "political_space": "Partido Justicialista",
            "mandate_type": "Provisional",
            "presidential_term": 1,
            "source_url": "https://www.casarosada.gob.ar/nuestro-pais/presidentes/47087-eduardo-oscar-camano-2001",
        },
        {
            "president": "Eduardo Duhalde",
            "valid_from": "2002-01-02",
            "valid_to_exclusive": "2003-05-25",
            "political_space": "Partido Justicialista",
            "mandate_type": "Designado por Asamblea Legislativa",
            "presidential_term": 1,
            "source_url": "https://www.casarosada.gob.ar/nuestro-pais/presidentes/47086-eduardo-alberto-duhalde-2002-2003",
        },
        {
            "president": "Néstor Kirchner",
            "valid_from": "2003-05-25",
            "valid_to_exclusive": "2007-12-10",
            "political_space": "Frente para la Victoria / PJ",
            "mandate_type": "Constitucional",
            "presidential_term": 1,
            "source_url": "https://www.casarosada.gob.ar/nuestro-pais/presidentes/47085-nestor-carlos-kirchner-2003-2007",
        },
        {
            "president": "Cristina Fernández de Kirchner",
            "valid_from": "2007-12-10",
            "valid_to_exclusive": "2011-12-10",
            "political_space": "Frente para la Victoria / PJ",
            "mandate_type": "Constitucional",
            "presidential_term": 1,
            "source_url": "https://www.casarosada.gob.ar/nuestro-pais/presidentes/47083-cristina-fernandez-de-kirchner-2007-2011",
        },
        {
            "president": "Cristina Fernández de Kirchner",
            "valid_from": "2011-12-10",
            "valid_to_exclusive": "2015-12-10",
            "political_space": "Frente para la Victoria / PJ",
            "mandate_type": "Constitucional",
            "presidential_term": 2,
            "source_url": "https://www.casarosada.gob.ar/nuestro-pais/presidentes/47082-cristina-fernandez-de-kirchner-2011-2015",
        },
        {
            "president": "Mauricio Macri",
            "valid_from": "2015-12-10",
            "valid_to_exclusive": "2019-12-10",
            "political_space": "Cambiemos / PRO",
            "mandate_type": "Constitucional",
            "presidential_term": 1,
            "source_url": "https://www.casarosada.gob.ar/nuestro-pais/presidentes/47081-mauricio-macri-2015-2019",
        },
        {
            "president": "Alberto Fernández",
            "valid_from": "2019-12-10",
            "valid_to_exclusive": "2023-12-10",
            "political_space": "Frente de Todos / PJ",
            "mandate_type": "Constitucional",
            "presidential_term": 1,
            "source_url": "https://www.argentina.gob.ar/defensa/nomina-de-presidentes-y-ministros",
        },
        {
            "president": "Javier Milei",
            "valid_from": "2023-12-10",
            "valid_to_exclusive": (end_date + timedelta(days=1)).isoformat(),
            "political_space": "La Libertad Avanza",
            "mandate_type": "Constitucional",
            "presidential_term": 1,
            "source_url": "https://www.argentina.gob.ar/defensa/nomina-de-presidentes-y-ministros",
        },
    ]

    return pd.DataFrame(records)

## 6. Transform: ministros del área económica

El CSV oficial usa `;` como separador, codificación Latin-1 y distintos separadores de fecha (`.`, `,` y `/`). La transformación normaliza esas variaciones.

El archivo incluye períodos en los que Hacienda y Finanzas fueron ministerios separados. Para obtener una única autoridad económica por día se conservan los cargos cuyo nombre contiene **Economía** o **Hacienda**. Un cargo exclusivamente denominado “Ministro de Finanzas” no se considera el responsable principal de la cartera económica durante el período de superposición.


In [ ]:
def strip_accents(value: str) -> str:
    """Convierte texto acentuado en una versión comparable sin acentos."""
    normalized = unicodedata.normalize("NFKD", str(value))
    return "".join(char for char in normalized if not unicodedata.combining(char))


def parse_source_date(value: Any) -> pd.Timestamp:
    """Interpreta las distintas variantes de fecha presentes en el CSV."""
    if pd.isna(value) or not str(value).strip():
        return pd.NaT

    cleaned = (
        str(value)
        .strip()
        .replace(",", ".")
        .replace("/", ".")
    )
    return pd.to_datetime(cleaned, dayfirst=True, errors="coerce")


def normalize_person_name(value: str) -> str:
    """Convierte 'APELLIDO, Nombres' en 'Nombres Apellido'."""
    value = " ".join(str(value).strip().split())

    if "," in value:
        surname, given_names = [part.strip() for part in value.split(",", 1)]
        value = f"{given_names} {surname}"

    return value.title()


def transform_economy_ministers(
    csv_bytes: bytes,
    start_date: date = START_DATE,
    end_date: date = END_DATE,
) -> pd.DataFrame:
    """Normaliza el CSV y construye intervalos continuos de responsabilidad."""
    raw = pd.read_csv(
        io.BytesIO(csv_bytes),
        sep=";",
        encoding="latin-1",
        dtype="string",
    )

    # Se renombran por posición para tolerar pequeñas variaciones de encoding
    # en los encabezados publicados por la fuente.
    raw = raw.iloc[:, :6].copy()
    raw.columns = [
        "ministry_title",
        "economy_minister",
        "source_valid_from",
        "appointment_norm_url",
        "source_valid_to",
        "departure_norm_url",
    ]

    title_comparable = raw["ministry_title"].map(strip_accents).str.lower()
    is_primary_economic_role = title_comparable.str.contains(
        r"economia|hacienda",
        regex=True,
        na=False,
    )

    df = raw.loc[is_primary_economic_role].copy()
    df["economy_minister"] = df["economy_minister"].map(normalize_person_name)
    df["valid_from"] = df["source_valid_from"].map(parse_source_date)
    df["source_valid_to"] = df["source_valid_to"].map(parse_source_date)

    df = (
        df.dropna(subset=["valid_from", "economy_minister"])
        .sort_values("valid_from")
        .reset_index(drop=True)
    )

    # Para el modelo diario, cada gestión se considera vigente hasta el inicio
    # de la siguiente. Se conserva source_valid_to para auditoría.
    df["valid_to_exclusive"] = df["valid_from"].shift(-1)
    df.loc[df.index[-1], "valid_to_exclusive"] = pd.Timestamp(end_date) + pd.Timedelta(days=1)

    df["transition_gap_days"] = (
        df["valid_to_exclusive"] - df["source_valid_to"]
    ).dt.days

    range_start = pd.Timestamp(start_date)
    range_end_exclusive = pd.Timestamp(end_date) + pd.Timedelta(days=1)

    df = df.loc[
        (df["valid_to_exclusive"] > range_start)
        & (df["valid_from"] < range_end_exclusive)
    ].copy()

    # Recortamos los intervalos al rango analizado.
    df["valid_from"] = df["valid_from"].clip(lower=range_start)
    df["valid_to_exclusive"] = df["valid_to_exclusive"].clip(
        upper=range_end_exclusive
    )
    df["extracted_at_utc"] = pd.Timestamp.now(tz="UTC")

    columns = [
        "economy_minister",
        "ministry_title",
        "valid_from",
        "valid_to_exclusive",
        "source_valid_to",
        "transition_gap_days",
        "appointment_norm_url",
        "departure_norm_url",
        "extracted_at_utc",
    ]
    return df[columns].reset_index(drop=True)

## 7. Transform: presidentes

La transformación convierte las fechas curadas a tipos temporales, recorta el primer mandato al año 2000 e incorpora una clave estable por período presidencial.


In [ ]:
def transform_presidential_terms(
    df: pd.DataFrame,
    start_date: date = START_DATE,
    end_date: date = END_DATE,
) -> pd.DataFrame:
    """Normaliza y recorta los períodos presidenciales al rango requerido."""
    result = df.copy()
    result["valid_from"] = pd.to_datetime(result["valid_from"])
    result["valid_to_exclusive"] = pd.to_datetime(result["valid_to_exclusive"])

    range_start = pd.Timestamp(start_date)
    range_end_exclusive = pd.Timestamp(end_date) + pd.Timedelta(days=1)

    result = result.loc[
        (result["valid_to_exclusive"] > range_start)
        & (result["valid_from"] < range_end_exclusive)
    ].copy()

    result["valid_from"] = result["valid_from"].clip(lower=range_start)
    result["valid_to_exclusive"] = result["valid_to_exclusive"].clip(
        upper=range_end_exclusive
    )
    result["presidential_term_id"] = (
        result["president"].str.lower().str.replace(r"[^a-záéíóúñ]+", "_", regex=True)
        + "_"
        + result["valid_from"].dt.strftime("%Y%m%d")
    )
    result["extracted_at_utc"] = pd.Timestamp.now(tz="UTC")

    return result.sort_values("valid_from").reset_index(drop=True)

## 8. Construcción de la dimensión diaria

El calendario diario transforma las tablas de intervalos en una dimensión con una fila por fecha. Este diseño hace que la tercera notebook pueda utilizar un join exacto por `date`, sin implementar un range join.

El día de inicio pertenece a la nueva autoridad. Por ejemplo, el 10 de diciembre de 2023 se asigna a Javier Milei y Luis Caputo.


In [ ]:
def build_daily_government_context(
    presidents: pd.DataFrame,
    ministers: pd.DataFrame,
    start_date: date = START_DATE,
    end_date: date = END_DATE,
) -> pd.DataFrame:
    """Expande los intervalos presidenciales y ministeriales a granularidad diaria."""
    daily = pd.DataFrame(
        {"date": pd.date_range(start=start_date, end=end_date, freq="D")}
    )

    president_columns = [
        "president",
        "political_space",
        "mandate_type",
        "presidential_term",
        "presidential_term_id",
    ]
    minister_columns = [
        "economy_minister",
        "ministry_title",
    ]

    for column in president_columns + minister_columns:
        daily[column] = pd.NA

    # Intervalos cerrados por izquierda y abiertos por derecha.
    for row in presidents.itertuples(index=False):
        mask = (
            (daily["date"] >= row.valid_from)
            & (daily["date"] < row.valid_to_exclusive)
        )
        for column in president_columns:
            daily.loc[mask, column] = getattr(row, column)

    for row in ministers.itertuples(index=False):
        mask = (
            (daily["date"] >= row.valid_from)
            & (daily["date"] < row.valid_to_exclusive)
        )
        for column in minister_columns:
            daily.loc[mask, column] = getattr(row, column)

    daily["president_change_flag"] = (
        daily["president"].ne(daily["president"].shift())
    )
    daily["economy_minister_change_flag"] = (
        daily["economy_minister"].ne(daily["economy_minister"].shift())
    )
    daily["government_change_flag"] = (
        daily["president_change_flag"]
        | daily["economy_minister_change_flag"]
    )
    daily["government_context"] = (
        daily["president"].astype("string")
        + " | "
        + daily["economy_minister"].astype("string")
    )
    daily["generated_at_utc"] = pd.Timestamp.now(tz="UTC")

    return daily

## 9. Controles de calidad

Las reglas principales son:

- una sola fila por fecha;
- cobertura completa del calendario;
- ausencia de intervalos presidenciales o ministeriales inválidos;
- ninguna autoridad faltante en la dimensión Silver.


In [ ]:
def validate_terms(
    presidents: pd.DataFrame,
    ministers: pd.DataFrame,
) -> None:
    """Valida que todos los intervalos tengan una duración positiva."""
    for name, df in {
        "presidents": presidents,
        "ministers": ministers,
    }.items():
        if df.empty:
            raise ValueError(f"La tabla {name} está vacía.")
        if (df["valid_to_exclusive"] <= df["valid_from"]).any():
            raise ValueError(f"La tabla {name} contiene intervalos inválidos.")


def validate_daily_context(
    daily: pd.DataFrame,
    start_date: date = START_DATE,
    end_date: date = END_DATE,
) -> None:
    """Valida cobertura, unicidad y autoridades completas."""
    expected_rows = (end_date - start_date).days + 1

    if len(daily) != expected_rows:
        raise ValueError(
            f"Se esperaban {expected_rows} días y se obtuvieron {len(daily)}."
        )
    if daily["date"].duplicated().any():
        raise ValueError("Existen fechas duplicadas en la dimensión diaria.")
    if daily["president"].isna().any():
        missing = daily.loc[daily["president"].isna(), "date"]
        raise ValueError(f"Hay días sin presidente: {missing.head().tolist()}")
    if daily["economy_minister"].isna().any():
        missing = daily.loc[daily["economy_minister"].isna(), "date"]
        raise ValueError(f"Hay días sin ministro: {missing.head().tolist()}")

## 10. Load: Delta Lake

Las tres tablas son dimensiones pequeñas y se reconstruyen como snapshots completos. El `overwrite` mantiene el pipeline simple e idempotente, mientras que Delta Lake conserva el historial transaccional.


In [ ]:
def save_snapshot_as_delta(
    df: pd.DataFrame,
    data_path: str,
    storage_options: dict,
) -> None:
    """Guarda un snapshot Delta sin persistir el índice de pandas."""
    arrow_table = pa.Table.from_pandas(df, preserve_index=False)
    write_deltalake(
        data_path,
        arrow_table,
        mode="overwrite",
        storage_options=storage_options,
    )


def vacuum_delta_table(
    data_path: str,
    storage_options: dict,
    retention_hours: int = VACUUM_RETENTION_HOURS,
) -> list[str]:
    """Elimina archivos de datos obsoletos respetando siete días de retención."""
    delta_table = DeltaTable(data_path, storage_options=storage_options)
    deleted_files = delta_table.vacuum(
        retention_hours=retention_hours,
        dry_run=False,
        enforce_retention_duration=True,
    )
    print(f"VACUUM {data_path}: {len(deleted_files)} archivos eliminados")
    return deleted_files

## 11. Orquestación

La carga sólo comienza después de que las transformaciones y las validaciones terminan correctamente.


In [ ]:
def run_pipeline() -> dict[str, Any]:
    """Ejecuta el ETL completo y devuelve métricas de la corrida."""
    started_at = pd.Timestamp.now(tz="UTC")

    # EXTRACT
    ministers_csv = download_ministers_csv()
    presidents_raw = extract_presidential_terms()

    # TRANSFORM
    presidents = transform_presidential_terms(presidents_raw)
    ministers = transform_economy_ministers(ministers_csv)
    daily_context = build_daily_government_context(presidents, ministers)

    # QUALITY
    validate_terms(presidents, ministers)
    validate_daily_context(daily_context)

    # LOAD
    datasets = [
        (presidents, PRESIDENTIAL_TERMS_PATH),
        (ministers, ECONOMY_MINISTER_TERMS_PATH),
        (daily_context, GOVERNMENT_DAILY_PATH),
    ]

    for dataframe, path in datasets:
        save_snapshot_as_delta(dataframe, path, storage_options)
        vacuum_delta_table(path, storage_options)

    finished_at = pd.Timestamp.now(tz="UTC")

    return {
        "started_at": started_at,
        "finished_at": finished_at,
        "duration_seconds": (finished_at - started_at).total_seconds(),
        "presidential_terms": len(presidents),
        "economy_minister_terms": len(ministers),
        "daily_rows": len(daily_context),
        "first_date": daily_context["date"].min(),
        "last_date": daily_context["date"].max(),
    }

In [ ]:
pipeline_metrics = run_pipeline()
pd.Series(pipeline_metrics, name="value").to_frame()

## 12. Validación posterior a la carga

Se vuelven a leer las tablas desde MinIO para comprobar el resultado persistido.


In [ ]:
presidents_delta = DeltaTable(
    PRESIDENTIAL_TERMS_PATH,
    storage_options=storage_options,
).to_pandas()

ministers_delta = DeltaTable(
    ECONOMY_MINISTER_TERMS_PATH,
    storage_options=storage_options,
).to_pandas()

government_daily_delta = DeltaTable(
    GOVERNMENT_DAILY_PATH,
    storage_options=storage_options,
).to_pandas()

government_daily_delta["date"] = pd.to_datetime(
    government_daily_delta["date"]
)
government_daily_delta = government_daily_delta.sort_values("date")

summary = pd.DataFrame(
    {
        "metric": [
            "Mandatos presidenciales",
            "Gestiones económicas",
            "Filas diarias",
            "Primera fecha",
            "Última fecha",
            "Fechas duplicadas",
            "Presidentes nulos",
            "Ministros nulos",
        ],
        "value": [
            len(presidents_delta),
            len(ministers_delta),
            len(government_daily_delta),
            government_daily_delta["date"].min(),
            government_daily_delta["date"].max(),
            int(government_daily_delta["date"].duplicated().sum()),
            int(government_daily_delta["president"].isna().sum()),
            int(government_daily_delta["economy_minister"].isna().sum()),
        ],
    }
)

display(summary)
display(government_daily_delta.tail(10))

## 13. Exploración: duración de las gestiones económicas

La visualización permite comprobar rápidamente la duración relativa de cada gestión dentro del período analizado.


In [ ]:
minister_duration = (
    government_daily_delta.groupby("economy_minister", as_index=False)
    .agg(days_in_context=("date", "count"))
    .sort_values("days_in_context")
)

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(
    minister_duration["economy_minister"],
    minister_duration["days_in_context"],
    color="#2563EB",
)
ax.set_title("Días por ministro del área económica desde 2000")
ax.set_xlabel("Cantidad de días")
ax.set_ylabel("Ministro/a")
ax.grid(axis="x", alpha=0.25)
fig.tight_layout()
plt.show()

## 14. Contrato para la futura notebook de integración

La tabla `government_context_daily` tiene una sola fila por `date`. Por lo tanto, la futura notebook podrá realizar:

```python
dataset = usd_quotes.merge(
    government_context_daily,
    on="date",
    how="left",
    validate="many_to_one",
)
```

Luego podrá calcular, por presidente o ministro:

- cotización inicial y final;
- variación absoluta y porcentual;
- cantidad de días con observación;
- impacto acumulado por gestión;
- eventos de cambio de autoridad.

Ese dataset será la capa Gold que alimentará el waterfall chart. Para evitar conclusiones causales incorrectas, la visualización deberá presentarse como una atribución temporal por gestión, no como prueba de que una persona causó por sí sola la variación observada.


## 15. Aprendizajes demostrados

- Integración de una fuente pública en CSV.
- Manejo de encoding y fechas inconsistentes.
- Modelado de dimensiones temporales con intervalos no superpuestos.
- Construcción de una dimensión diaria para simplificar joins.
- Separación entre Bronze y Silver.
- Calidad de datos antes y después de la persistencia.
- Escritura idempotente y mantenimiento de tablas Delta.
- Diseño anticipado de un contrato de datos para una capa Gold y visualización posterior.
